# **PicassoPy Workshop --> Test case: cpv 2024-08-21**
---


- Folder for data `test_case_data` -> download separately
- Folder for configs `test_case_config`


## Imports

In [1]:
import os, sys
import argparse
import datetime
import logging
from pathlib import Path
import numpy as np

sys.path.append('../')
import ppcpy
import ppcpy.io.loadConfigs as loadConfigs
import ppcpy.io.readPollyRawData as readPollyRawData
import ppcpy.interface.picassoProc as picassoProc
import ppcpy.misc.helper as helper
import ppcpy.misc.startscreen as startscreen
from ppcpy.io.write2nc import write_channelwise_2_nc_file, write2nc_file, write_profile2nc_file

import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.tab20.colors)

## Defining Script Inputs

- The parameters `args.device`, `args.timestamp`, `args.picasso_config_file`, `args.level0_file_to_process`, need to be manually specified per case.
- The parameter `DATABASE_PATH` is the path to the database used for storing the retrieved calibration constants

In [2]:
## For purpose of the notebook mimic the argparse interface
from types import SimpleNamespace
args = SimpleNamespace()

## The used device and time of mesurment
args.device = 'pollyxt_cpv'
args.timestamp = '20240821'
dt = datetime.datetime.strptime(args.timestamp, "%Y%m%d")

## The used config file
args.picasso_config_file = "test_case_config/pollynet_processing_chain_config_test.json"

## The data file to use
args.level0_file_to_process = f"test_case_data/{dt:%Y_%m_%d_%a}_CPV_00_00_01.nc"


## Database path
DATABASE_PATH = "PicassoPyDatabase.db" # I should test if it works to read this directly from the config files.

In [3]:
startscreen.startscreen()

      ____  _                            ____           ___ ____ 
     / __ \(_)________ _______________  / __ \__  __   <  // __ \
    / /_/ / / ___/ __ `/ ___/ ___/ __ \/ /_/ / / / /   / // / / /
   / ____/ / /__/ /_/ (__  |__  ) /_/ / ____/ /_/ /   / // /_/ / 
  /_/   /_/\___/\__,_/____/____/\____/_/    \__, /   /_(_)____/  
                                           /____/                


## Load Data and Config-files

Loads data and information from the config files into the following three dictionaries:
- `picasso_config_dict`: Paths and other information stored in the picasso config file
- `polly_config_dict`: Configuration variables from polly config and polly default files
- `rawdata_dict`: Measurement data and information extracted from the level0 file

In [4]:
## Path to dafault Picasso config file
picasso_default_config_file = Path(
    helper.detect_path_type(Path.cwd().parent), 'ppcpy', 'config', 'pollynet_processing_chain_config.json')

## Load Picasso config file
picasso_config_dict = loadConfigs.loadPicassoConfig(args.picasso_config_file, picasso_default_config_file)

## load polly config file
polly_config_array = loadConfigs.readPollyNetConfigLinkTable(picasso_config_dict['pollynet_config_link_file'], timestamp=args.timestamp, device=args.device)
polly_config_dict = loadConfigs.getPollyConfigfromArray(
    polly_config_array, picasso_config_dict
)

## Load level0-data file
rawfile_fullname = args.level0_file_to_process
rawfile = helper.detect_path_type(rawfile_fullname)
rawdata_dict = readPollyRawData.readPollyRawData(rawfile)

2026-05-11 10:16:02,329 - INFO - picasso_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\pollynet_processing_chain_config.json
2026-05-11 10:16:02,331 - INFO - picasso_config_file: test_case_config/pollynet_processing_chain_config_test.json
2026-05-11 10:16:02,332 - INFO - pollynet_config_link_file: test_case_config/pollynet_processing_chain_config_links.xlsx
2026-05-11 10:16:02,507 - INFO - polly_default_config_file: c:\Users\buholdt\Documents\PicassoPy\ppcpy\config\polly_global_config.json
2026-05-11 10:16:02,508 - INFO - polly_config_file: test_case_config\pollyxt_cpv_config_20230927.json
2026-05-11 10:16:02,510 - INFO - keys default/template file, but not in specific file {'search_cloud_above', 'molDepol1064', 'bgCorRangeIndxHigh', 'zLim_VolDepol_532', 'flagUseManualRefH', 'polCaliEta532', 'xLim_Profi_LR', 'zLim_NR_RCS_407', 'turbid_thres_par_beta_1064', 'prodSaveList', 'refH_FR_532', 'spheroid_thres_par_depol', 'radiosondeType', 'maxCloudSearchHeight', 'isPa

## Initialize PicassoProc object

PicassoProc is the main object in the PicassoPy, and is responsible for running all processes included and storing the data.

In [5]:
## Initialize PicassoProc
data_cube = picassoProc.PicassoProc(rawdata_dict, polly_config_dict, picasso_config_dict)

In [6]:
## reset date if date in filename differs date within nc-file 
data_cube.reset_date_infile()

## checking for correct mshots
data_cube.check_for_correct_mshots()

## setting channelTags
data_cube.setChannelTags()

## check for correct date in nc-file
data_cube.reset_date_infile()

2026-05-11 10:16:02,801 - INFO - date consistency-check... 
2026-05-11 10:16:02,801 - INFO - ... date in nc-file equals date of filename
2026-05-11 10:16:02,802 - INFO - ChannelLabels: ['FR-total-355 nm', 'FR-cross-355 nm', 'FR-387 nm', 'FR-407 nm', 'FR-total-532 nm', 'FR-cross-532 nm', 'FR-607 nm', 'FR-total-1064 nm', 'NR-total-532 nm', 'NR-607 nm', 'NR-total-355 nm', 'NR-387 nm', 'DFOV', '1058', '1064s', 'none']
2026-05-11 10:16:02,804 - WARNING - removed none tag from channel list [15]
2026-05-11 10:16:02,805 - INFO - date consistency-check... 
2026-05-11 10:16:02,805 - INFO - ... date in nc-file equals date of filename


## Preprocessing & Saturation Detection

The preprocessing includes the following processes:
- Deadtime correction
- Background correction
- SNR claculations
- Flagging of data
- Range correction

In [7]:
## Perform preprocessing, this includes Dead-time correction, Background correction, and Range correction
data_cube.preprocessing(collect_debug=True)

2026-05-11 10:16:02,812 - INFO - starting data preprocessing...
2026-05-11 10:16:02,812 - INFO - ... time conversion
2026-05-11 10:16:02,818 - WARNING - ... mShots not constant min 2993 max 2999
2026-05-11 10:16:02,819 - INFO - ... Deadtime-correction (Mode: 1)


mShots_norm (720, 15) mShots_norm [2995.73333333 2995.73333333 2995.73333333 2995.73333333 2995.73333333
 2995.73333333 2995.73333333 2995.73333333 2995.72361111 2995.72361111
 2995.72361111 2995.72361111 2995.72361111 2995.72361111 2995.72361111]


2026-05-11 10:16:06,004 - INFO - ... removing background from signal
2026-05-11 10:16:06,511 - INFO - ... height bin calculations
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\preprocess\pollyPreprocess.py:653: UserWarning: no explicit representation of timezones available for np.datetime64
  data_dict['time64'] = np.array([np.datetime64(t) for t in mTime_obj])
2026-05-11 10:16:06,623 - INFO - ... mask bins with low SNR


flag 532 FR [False False False False  True False False False False False False False
 False False False]
flag 355 FR [ True False False False False False False False False False False False
 False False False]
flag 607 FR [False False False False False False  True False False False False False
 False False False]


2026-05-11 10:16:08,481 - INFO - ... mask for polarization calibration
2026-05-11 10:16:08,485 - INFO - ... calculate range-corrected Signal


flagNDepCal [False False False False False False False False False False False False
  True  True  True  True  True  True  True  True False]
flagPDepCal [False False  True  True  True  True  True  True  True  True False False
 False False False False False False False False False]
depCalPeriods [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

2026-05-11 10:16:08,843 - INFO - finished data preprocessing.


In [8]:
## Save high resolution signal-to-noise ratio, background, and range corrected signal
write_channelwise_2_nc_file(data_cube=data_cube, prod_ls=['SNR', 'BG', 'RCS'])

2026-05-11 10:16:08,883 - INFO - saving product: SNR
2026-05-11 10:16:09,000 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_SNR.nc
2026-05-11 10:16:11,327 - INFO - saving product: BG
2026-05-11 10:16:11,407 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_BG.nc
2026-05-11 10:16:11,423 - INFO - saving product: RCS
2026-05-11 10:16:11,503 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_RCS.nc


In [9]:
## Display available channels
data_cube.channel_dict

{0: 'FR-total-355 nm',
 1: 'FR-cross-355 nm',
 2: 'FR-387 nm',
 3: 'FR-407 nm',
 4: 'FR-total-532 nm',
 5: 'FR-cross-532 nm',
 6: 'FR-607 nm',
 7: 'FR-total-1064 nm',
 8: 'NR-total-532 nm',
 9: 'NR-607 nm',
 10: 'NR-total-355 nm',
 11: 'NR-387 nm',
 12: 'DFOV',
 13: '1058',
 14: '1064s'}

In [10]:
## Detect and flag saturated signal
data_cube.SaturationDetect()

2026-05-11 10:16:15,211 - INFO - Saturation detection


In [11]:
import ppcpy.io.sql_interaction as sql_db
table_name = 'depol_calibration_constant'
ts_interval = data_cube.retrievals_highres['time'][0], data_cube.retrievals_highres['time'][-1]
test = sql_db.get_from_sql_db(DATABASE_PATH, table_name, ts_interval)['D90_db']
test

{'1064_FR': [{'eta': 0.14153155793311498,
   'eta_std': 0.008380733632373237,
   'time_start': 1724207790,
   'time_end': 1724208000}],
 '355_FR': [{'eta': 47.742418057789656,
   'eta_std': 1.0396617649540707,
   'time_start': 1724207790,
   'time_end': 1724208000}],
 '532_FR': [{'eta': 12.380587648929659,
   'eta_std': 0.13602867159593543,
   'time_start': 1724207790,
   'time_end': 1724208000}]}

## Depol Calibration


- Depol. calibration constants (DC) are retrieved at each depol. calibration period included in the data
- All retrieved DCs are stored in a dedicated database
- The optimal retrieved DC, ie. the one with the lowest standard deviation (std) is used for the processing
- If no DCs can be retirieved, the DC with the lowest std in the time range [24h before the measurement, 24h after the measurement] included in the database will be used

In [12]:
## Delta 90 polarization calibration
data_cube.polarizationCaliD90()

2026-05-11 10:16:18,051 - INFO - and even a 355 channel
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:314: RuntimeWarning: divide by zero encountered in divide
  dplus = smooth_signal(sig_x_p, smooth_win) / smooth_signal(sig_t_p, smooth_win)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:314: RuntimeWarning: invalid value encountered in divide
  dplus = smooth_signal(sig_x_p, smooth_win) / smooth_signal(sig_t_p, smooth_win)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\polarization.py:315: RuntimeWarning: divide by zero encountered in divide
  dminus = smooth_signal(sig_x_m, smooth_win) / smooth_signal(sig_t_m, smooth_win)
2026-05-11 10:16:18,124 - INFO - pol_cali_355  [{'eta': 46.770013651088924, 'eta_std': 0.9418544378987289, 'time_start': 1724207790, 'time_end': 1724208000, 'status': 1}]
2026-05-11 10:16:18,125 - INFO - and even a 532 channel
2026-05-11 10:16:18,200 - INFO - pol_cali_532  [{'eta': 1

starting loadGHK
data_cube keys  dict_keys(['rawfile', 'rawdata_dict', 'polly_config_dict', 'picasso_config_dict', 'device', 'location', 'date', 'num_of_channels', 'num_of_profiles', 'retrievals_highres', 'retrievals_profile', 'pol_cali', 'LC', 'channel_dict', 'flags', 'flag_355_total_FR', 'flag_355_cross_FR', 'flag_355_parallel_FR', 'flag_355_total_NR', 'flag_387_total_FR', 'flag_387_total_NR', 'flag_407_total_FR', 'flag_407_total_NR', 'flag_532_total_FR', 'flag_532_cross_FR', 'flag_532_parallel_FR', 'flag_532_total_NR', 'flag_532_cross_DFOV', 'flag_532_rr_FR', 'flag_607_total_FR', 'flag_607_total_NR', 'flag_1058_total_FR', 'flag_1064_total_FR', 'flag_1064_cross_FR', 'flag_1064_total_NR', 'flagSaturation'])
Using GHK from config file
G [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
H [ 0.02041 -0.998    1.       1.      -0.01477 -0.9987   1.      -0.02439
  1.       1.       1.       1.       1.       1.      -0.996  ]
K [0.97859 1.      1.      1.      0.99343 1.      1.      0.9947 

2026-05-11 10:16:18,266 - INFO - pol_cali_1064  [{'eta': 0.1511511338177992, 'eta_std': 0.0049707222337296636, 'time_start': 1724207790, 'time_end': 1724208000, 'status': 1}]
2026-05-11 10:16:18,268 - INFO - Using retieved polarization calibration constants.


[{'eta': 0.1511511338177992, 'eta_std': 0.0049707222337296636, 'time_start': 1724207790, 'time_end': 1724208000, 'status': 1}]


In [13]:
## Display depolarization calibration constants
data_cube.etaused

{'355_FR': np.float64(46.770013651088924),
 '532_FR': np.float64(12.397702207784498),
 '1064_FR': np.float64(0.1511511338177992)}

## Cloud Screening

Three modes of cloud Screening are currently implemented:

0. No cloud screening. Return cloud free for all timestamps
1. Cloud screen with Maximum Gradiant Signal (MSG) algorithm
2. Cloud screen with Zhao's algorithm

Clouds are screened per timestamp (30s). After the screening the data is splitt up into cloud free segments and aggregated.

In [14]:
## Apply cloud screening
data_cube.cloudScreen()

2026-05-11 10:16:18,290 - INFO - cloud screen mode 1: MSG method.


Starting cloud screen


In [15]:
## Segmentate cloud free groups
data_cube.cloudFreeSeg()

intNProfiles 120 minIntNProfiles 30


In [16]:
## Display cloud free groups
data_cube.clFreeGrps

array([[  0, 113],
       [144, 244]])

In [17]:
## Aggregate background, background corrected signal, and range corrected signal
data_cube.aggregate_profiles()

## Molecular Profiles

The molecular profiles are calculated from cloudNet ECMWF model data. Gdas1 data is not supported in PicassoPy!

In [18]:
## QuickFix for loadMeteo bug:
METEO_DATA_DIR_PATH = "E:\\data\\level1a\\cloudnet\\mindelo\\calibrated\\ecmwf"
data_cube.polly_config_dict['meteorDataSource'] = 'nc_cloudnet'
data_cube.polly_config_dict['meteo_folder'] = METEO_DATA_DIR_PATH
data_cube.polly_config_dict['meteo_file'] = r"[\\/]{0:%Y}[\\/]{0:%Y%m%d}_.*\.nc"

## Load meteorological data
data_cube.loadMeteo()

## Calculate molecular profiles
data_cube.calcMolecular()

c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\io\readMeteo.py:263: FutureWarning: In a future version, xarray will not decode the variable 'forecast_time' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  ds = xr.load_dataset(filename)
2026-05-11 10:16:19,785 - INFO - Performing model height correction.
2026-05-11 10:16:19,786 - INFO - Uncorrected model height[:,0]:
[10.466079  10.4652    10.460253  10.455192  10.449221  10.447984
 10.453651  10.460172  10.464973  10.472917  10.479133  10.485509
 10.4861145 10.477216  10.487477  10.491782  10.485256  10.4855

time slices of cloud free  [array(['2024-08-21T00:00:00.000000', '2024-08-21T00:56:30.000000'],
      dtype='datetime64[us]'), array(['2024-08-21T01:12:00.000000', '2024-08-21T02:02:00.000000'],
      dtype='datetime64[us]')]
get_mean_profiles(time_slice: <class 'list'>) -> <class 'list'>
len mean_profiles 2
shape of the molecular scattering (2, 4000)
for the wavelengths  [355, 387, 407, 532, 607, 1058, 1064]


## Rayleigh-Fit

- Douglas-Peucker algorithm is used to segment the signal into potential reference heights
- The reference height with the best fit to the molecular backscatter per channel is chosen
- Currently only done for FR-channels. NR reference heights are read from the config variables `refH_NR_{wavelength}`

In [19]:
## Rayleigh-fit procedure --> produces the reference heights
data_cube.rayleighFit()

2026-05-11 10:16:19,854 - WARNING - Potential for differences to matlab code due to numerical issues (subtraction of two small values)
2026-05-11 10:16:19,855 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:19,856 - WARNING - at 10km height this is a difference of about 4 indices
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\calibration\rayleighfit.py:571: RuntimeWarning: divide by zero encountered in divide
  std_aer_norm = sig_aer_norm / np.sqrt(pc + bg)


Start Rayleigh Fit
0 [  0 113]
refH for 532
DPInd: [ 107  508  509  910  911 1312 1313 1714 1715 2116 2117 2408]
refInd: (1715, 2116)
refH for 355
DPInd: [ 120  922  923 1725 1726 2408]
refInd: (923, 1725)
refH for 1064
DPInd: [ 107  374  375  642  643  689  956  957 1224 1225 1492 1493 1760 1761
 1816 1839 1848 1849 1872 1945 1956 1960 2005 2058 2066 2070 2073 2090
 2091 2099 2127 2128 2139 2142 2147 2160 2161 2172 2193 2321 2363 2397
 2398 2411 2438 2509 2512 2537 2545 2548 2554 2586 2605 2606 2609 2624
 2691 2797 2848 2860 2904 2908 2915 2928 2929 2937 2954 2956 2959 2984
 3001 3003 3004 3032 3040 3043 3047 3048 3049 3062 3065 3074 3086 3088
 3096 3111 3160 3185 3193 3211 3216 3298 3311 3321 3325 3333 3335 3343]
one tests failed?
refInd: (nan, nan)
1 [144 244]
refH for 532
DPInd: [ 107  508  509  910  911 1312 1313 1714 1715 2116 2117 2408]
refInd: (1313, 1714)
refH for 355
DPInd: [ 120  922  923 1725 1726 2406]
refInd: (1726, 2406)
refH for 1064
DPInd: [ 107  374  375  642  643  69

2026-05-11 10:16:20,092 - INFO - Using Config values for NR refH.


one tests failed?
one tests failed?
refInd: (1940, 2071)


In [20]:
## Display reference heights for a given cloud free group
grpIdx = 0
print(f"""Reference heights in meters for cloud free period {grpIdx} {data_cube.retrievals_highres['time64'][data_cube.clFreeGrps[grpIdx]]}:
355 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_FR"]['refHeight'], 0)}
532 total FR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_FR"]['refHeight'], 0)}
1064 total FR: {np.round(data_cube.retrievals_profile['refH'][grpIdx]["1064_total_FR"]['refHeight'], 0)}
355 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["355_total_NR"]['refHeight'], 0)}
532 total NR:  {np.round(data_cube.retrievals_profile['refH'][grpIdx]["532_total_NR"]['refHeight'], 0)}""")

Reference heights in meters for cloud free period 0 ['2024-08-21T00:00:00.000000' '2024-08-21T00:56:30.000000']:
355 total FR:  [ 6877. 12849.]
532 total FR:  [12775. 15761.]
1064 total FR: [nan nan]
355 total NR:  [3005. 3995.]
532 total NR:  [3005. 3995.]


## GHK-Transmission Correction

In [21]:
## Molecular polarization calibration 
data_cube.polarizationCaliMol()

2026-05-11 10:16:20,116 - WARNING - not checked against the matlab code
2026-05-11 10:16:20,116 - WARNING - 'flagMolDepolCali' set to False


In [22]:
## Apply GHK-transmission correction
data_cube.transCor()

2026-05-11 10:16:20,126 - WARNING - transmission correction
2026-05-11 10:16:20,178 - INFO - and even a 355 channel


G [1.] [1.]
H [0.02041] [-0.998]
polCaliEta 46.770013651088924
G [1.] [1.] H [0.02041] [-0.998] Eta 46.770013651088924 error [ 0.00016  0.01567 -0.00958] Window 1 
calculated R_t [0.95999647]


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:168: RuntimeWarning: divide by zero encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:168: RuntimeWarning: invalid value encountered in divide
  sig_ratio = sigc / sigt
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\depolarization.py:178: RuntimeWarning: invalid value encountered in divide
  vol_depol = (sig_ratio / eta * (Gt + Ht) - (Gr + Hr)) / ((Gr - Hr) - sig_ratio / eta * (Gt - Ht))
2026-05-11 10:16:20,327 - INFO - and even a 532 channel


G [1.] [1.]
H [-0.01477] [-0.9987]
polCaliEta 12.397702207784498
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 
calculated R_t [1.02998285]


2026-05-11 10:16:20,491 - INFO - and even a 1064 channel


G [1.] [1.]
H [-0.02439] [-0.996]
polCaliEta 0.1511511338177992
G [1.] [1.] H [-0.02439] [-0.996] Eta 0.1511511338177992 error [ 0.00133  0.02379 -0.01205] Window 1 
calculated R_t [1.04999949]


In [23]:
## Aggregate GHK-transmission corrected profiles
data_cube.aggregate_profiles(var='sigTCor')
data_cube.aggregate_profiles(var='BGTCor')

## Klett and Raman retrieval

Produces the following profiles per channel:
- Klett: Aerosol Backscatter and Extinction
- Raman: Aerosol Backscatter, Aerosol Extinction, and Lidar Ratio

If `nr=True`, perform the retrievals for FR and NR channels. Otherwise only FR.

In [24]:
## Klett retrieval for GHK-transmisson corrected profiles
data_cube.retrievalKlett(nr=True)

2026-05-11 10:16:20,712 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:20,713 - WARNING - at 10km height this is a difference of about 4 indices


retrievalname klett
Starting Klett retrieval
cldFree  0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
== 532, total, FR klett =================================
== 355, total, FR klett =================================
== 1064, total, FR klett =================================
No valid refHInd found, skipping Klett retrieval for this channel.
== 532, total, NR klett =================================
== 355, total, NR klett =================================
cldFree  1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
== 532, total, FR klett =================================
== 355, total, FR klett =================================
== 1064, total, FR klett =================================
Signal is too noisy at the reference height, skipping Klett retrival for this channel. [0.] 0.5
== 532, total, NR klett =================================
== 355, total, NR klett =================================


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:311: RuntimeWarning: invalid value encountered in divide
  aerRelBRStd = np.abs((1 + noise / signal) / (1 + aerBsc + molBsc / 1e3) - 1)


In [25]:
## Raman retrieval for GHK-transmisson corrected profiles
data_cube.retrievalRaman(nr=True)

2026-05-11 10:16:20,929 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:20,930 - WARNING - at 10km height this is a difference of about 4 indices


[array([900, 900, 900, 900, 800, 800, 800, 800, 150, 150, 150, 150, 150,
       800, 800]), array([900, 900, 900, 900, 800, 800, 800, 800, 150, 150, 150, 150, 150,
       800, 800])]
Starting Raman retrieval
cldFree  0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
== 355, total, FR | 387, total, FR raman ========


c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


900 370.01248
refHInd (923, 1725) refH [ 6903.17491198 12898.12483549] hBaseInd 170 hBase 1274.4999837875366
filling aerExt below overlap with 0.00011922955234656786 for calculating the backscatter


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\raman.py:525: RuntimeWarning: invalid value encountered in sqrt
  sigElasticSample = sigGenWithNoise(sigElastic, np.sqrt(sigElastic + bgElastic), MC_count[2], 'norm').T


== 532, total, FR | 607, total, FR raman ========
800 370.01248
refHInd (1715, 2116) refH [12823.37483644 15820.8497982 ] hBaseInd 157 hBase 1177.3249850273132
filling aerExt below overlap with 0.00011280231294580689 for calculating the backscatter
== 1064, total, FR | 607, total, FR raman ========
No valid refHInd found, skipping Raman retrieval for this channel.
== 532, total, NR | 607, total, NR raman ========
150 190.6125
refHInd [403, 536] refH [3016.17496157 4010.34994888] hBaseInd 46 hBase 347.59999561309814
filling aerExt below overlap with 0.00018262783244339073 for calculating the backscatter


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\raman.py:362: RuntimeWarning: invalid value encountered in sqrt
  noise = np.sqrt(sig + bg)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\raman.py:526: RuntimeWarning: invalid value encountered in sqrt
  sigVRN2Sample = sigGenWithNoise(sigVRN2, np.sqrt(sigVRN2 + bgVRN2), MC_count[2], 'norm').T


== 355, total, NR | 387, total, NR raman ========
150 190.6125
refHInd [403, 536] refH [3016.17496157 4010.34994888] hBaseInd 46 hBase 347.59999561309814
filling aerExt below overlap with 0.00018223952549901464 for calculating the backscatter
cldFree  1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
== 355, total, FR | 387, total, FR raman ========
900 370.01248
refHInd (1726, 2406) refH [12905.5998354  17988.59977055] hBaseInd 170 hBase 1274.4999837875366
filling aerExt below overlap with 9.906391498815921e-05 for calculating the backscatter
== 532, total, FR | 607, total, FR raman ========
800 370.01248
refHInd (1313, 1714) refH [ 9818.42487478 12815.89983654] hBaseInd 157 hBase 1177.3249850273132
filling aerExt below overlap with 9.485055472829043e-05 for calculating the backscatter
== 1064, total, FR | 607, total, FR raman ========
800 370.01248
refHInd (1940, 2071) refH [14505.24981499 15484.47480249] hBaseInd 157 hBase 1177.3249850273132
filling aerExt below overlap with 7.

## Overlap Correction

Two methods are available for calculating the Overlap Function:

1. FRNR method
2. Raman method

And four methods (currently only 3 implemented) are available for applying the Overlap Correction:

0. no overlap correction
1. overlap correction with using the default overlap function (read function from file)
2. overlap correction with using the calculated overlap function
3. overlap correction with gluing near-range and far-range signal -> Not implemented yet!


In [26]:
## Calculate overlap function
data_cube.overlapCalc()

## Fix spike in lower bins
data_cube.overlapFixLowestBins()

## Apply overlap correction
data_cube.overlapCor()

2026-05-11 10:16:25,800 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:25,801 - WARNING - at 10km height this is a difference of about 4 indices


Starting Overlap retrieval
cldFree 0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
355 both telescopes available
387 both telescopes available
532 both telescopes available
607 both telescopes available
cldFree 1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
355 both telescopes available


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\helper.py:910: RuntimeWarning: invalid value encountered in scalar divide
  relStd.append(thisStd / abs(thisMean))


387 both telescopes available
532 both telescopes available
607 both telescopes available


2026-05-11 10:16:27,632 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:27,633 - WARNING - at 10km height this is a difference of about 4 indices


Starting Raman Overlap retrieval
cldFree  0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
532 607 both wavelengths available
355 387 both wavelengths available
cldFree  1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
532 607 both wavelengths available
355 387 both wavelengths available


2026-05-11 10:16:31,173 - INFO - overlap Correction


Fixing lower bins for frnr overlap functions
Fixing lower bins for raman overlap functions
overlapCorMode  2  overlapCalMode  2
dict_keys(['frnr', 'raman'])
overlap correction source raman
[array(['2024-08-21T00:00:00.000000', '2024-08-21T00:56:30.000000'],
      dtype='datetime64[us]'), array(['2024-08-21T01:12:00.000000', '2024-08-21T02:02:00.000000'],
      dtype='datetime64[us]')] ['2024-08-21T00:00:00.000000' '2024-08-21T00:56:30.000000'
 '2024-08-21T01:12:00.000000' '2024-08-21T02:02:00.000000']
[[  0 113]
 [144 244]]
355_total_FR len(olFuncs) 2 2 2
(4, 4000)
(720, 4000)
532_total_FR len(olFuncs) 2 2 2
(4, 4000)
(720, 4000)
correct overlap 355
correct overlap 387
using 355 instead of 387
correct overlap 532
correct overlap 607
using 532 instead of 607
correct overlap 1064
using 532 instead of 1064


In [27]:
## Aggregate overlap corrected profiles
data_cube.aggregate_profiles('sigOLCor')
data_cube.aggregate_profiles('BGOLCor')

In [28]:
## Klett retrieval for overlap corrected profiles
data_cube.retrievalKlett(oc=True)

## Raman retrieval for overlap corrected profiles
data_cube.retrievalRaman(oc=True)

2026-05-11 10:16:31,531 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:31,532 - WARNING - at 10km height this is a difference of about 4 indices


retrievalname klett_OC
Starting Klett retrieval
cldFree  0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
== 532, total, FR klett =================================
== 355, total, FR klett =================================
== 1064, total, FR klett =================================
No valid refHInd found, skipping Klett retrieval for this channel.
cldFree  1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
== 532, total, FR klett =================================
== 355, total, FR klett =================================


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:277: RuntimeWarning: invalid value encountered in scalar divide
  denominator1 = RCS[iAlt + 1] / (aerBsc[iAlt + 1] + molBsc[iAlt + 1])
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\klettfernald.py:311: RuntimeWarning: divide by zero encountered in divide
  aerRelBRStd = np.abs((1 + noise / signal) / (1 + aerBsc + molBsc / 1e3) - 1)
2026-05-11 10:16:31,645 - WARNING - rayleighfit seems to use range in matlab, but the met data should be in height >> RECHECK!
2026-05-11 10:16:31,646 - WARNING - at 10km height this is a difference of about 4 indices


== 1064, total, FR klett =================================
[array([145., 900., 145., 900., 130., 800., 130., 130., 150., 150., 150.,
       150., 150., 800., 800.]), array([145., 900., 145., 900., 138., 800., 138., 138., 150., 150., 150.,
       150., 150., 800., 800.])]
Starting Raman retrieval
cldFree  0 [  0 113]
cldFree mod (np.int64(0), np.int64(114))
== 355, total, FR | 387, total, FR raman ========


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\raman.py:339: RuntimeWarning: divide by zero encountered in divide
  temp = number_density / (sig * height**2)


145.0 370.01248
refHInd (923, 1725) refH [ 6903.17491198 12898.12483549] hBaseInd 69 hBase 519.5249934196472
filling aerExt below overlap with 0.00022646820368069827 for calculating the backscatter
== 532, total, FR | 607, total, FR raman ========
130.0 370.01248
refHInd (1715, 2116) refH [12823.37483644 15820.8497982 ] hBaseInd 67 hBase 504.5749936103821
filling aerExt below overlap with 0.00010405411216919048 for calculating the backscatter
== 1064, total, FR | 607, total, FR raman ========
No valid refHInd found, skipping Raman retrieval for this channel.
cldFree  1 [144 244]
cldFree mod (np.int64(144), np.int64(245))
== 355, total, FR | 387, total, FR raman ========
145.0 370.01248
refHInd (1726, 2406) refH [12905.5998354  17988.59977055] hBaseInd 69 hBase 519.5249934196472
filling aerExt below overlap with 0.00023221767859718214 for calculating the backscatter
== 532, total, FR | 607, total, FR raman ========
138.0 370.01248
refHInd (1313, 1714) refH [ 9818.42487478 12815.89983654

## Depol and Ångström Profiles

Retrieval of Volume and Particle depolarization as well as Ångström 355/532 backscatter 532/1064 backscatter, and 355/532 Extinction

In [29]:
## Volume and particle depolarization
data_cube.calcDepol()

2026-05-11 10:16:34,473 - INFO - voldepol at channel 532 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,476 - INFO - voldepol at channel 355 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,479 - INFO - voldepol at channel 532 cldFree 1 (np.int64(144), np.int64(245))
2026-05-11 10:16:34,481 - INFO - voldepol at channel 355 cldFree 1 (np.int64(144), np.int64(245))
2026-05-11 10:16:34,484 - INFO - pardepol at channel 532 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,486 - INFO - pardepol at channel 355 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,488 - INFO - pardepol at channel 532 cldFree 1 (np.int64(144), np.int64(245))
2026-05-11 10:16:34,490 - INFO - pardepol at channel 355 cldFree 1 (np.int64(144), np.int64(245))
2026-05-11 10:16:34,491 - INFO - voldepol at channel 355 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,494 - INFO - voldepol at channel 532 cldFree 0 (np.int64(0), np.int64(114))
2026-05-11 10:16:34,496 - INFO -

klett
no_profiles  2
dict_keys(['532_total_FR', '355_total_FR', '532_total_NR', '355_total_NR'])
532_total_FR 12.397702207784498
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 25 
est. mdr 532_total_FR  0.005274405552121577 0.000377379508040164
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 
est. mdr 532_total_FR  0.0048645647496549215 0.0003818298365559713  (smooth1)
dict_keys(['aerBsc', 'aerBscStd', 'aerBR', 'aerBRStd', 'aerExt', 'aerExtStd', 'retrieval', 'signal', 'refBeta', 'vdr', 'vdrStd', 'mdr', 'mdrStd'])
355_total_FR 46.770013651088924
G [1.] [1.] H [0.02041] [-0.998] Eta 46.770013651088924 error [ 0.00016  0.01567 -0.00958] Window 25 
est. mdr 355_total_FR  0.007660594196629387 0.00028097258056102687
G [1.] [1.] H [0.02041] [-0.998] Eta 46.770013651088924 error [ 0.00016  0.01567 -0.00958] Window 1 
est. mdr 355_total_FR  0.008913544943766444 0.000303484537585882  (smoot

In [30]:
## Ångström ratios
data_cube.Angstroem()

klett
channels available 355_total_FR 532_total_FR Bsc
channels available 355_total_NR 532_total_NR Bsc
channels available 355_total_FR 532_total_FR Ext
channels available 355_total_NR 532_total_NR Ext
channels available 355_total_FR 532_total_FR Bsc
channels available 355_total_NR 532_total_NR Bsc
channels available 355_total_FR 532_total_FR Ext
channels available 355_total_NR 532_total_NR Ext
raman
channels available 355_total_FR 532_total_FR Bsc
channels available 355_total_NR 532_total_NR Bsc
channels available 355_total_FR 532_total_FR Ext
channels available 355_total_NR 532_total_NR Ext
channels available 355_total_FR 532_total_FR Bsc
channels available 355_total_NR 532_total_NR Bsc
channels available 532_total_FR 1064_total_FR Bsc
channels available 355_total_FR 532_total_FR Ext
channels available 355_total_NR 532_total_NR Ext
klett_OC
channels available 355_total_FR 532_total_FR Bsc
channels available 355_total_FR 532_total_FR Ext
channels available 355_total_FR 532_total_FR Bs

## Lidar Calibration

- Lidar calibration constants (LC) are retieved for each channel at each cloud free period for both Klett and Raman retieved profiles
- All retrieved LCs are stored in the database
- If no LCs can be retrieved for a given channel, the LCs in the time range [24h before the measurment, 24h after the measurement] for the given channel included in the database will be used
- The optimal LC, ie. the one with the lowest standard deviation per channel is used for the processing.
- The following priority order is used when choosing the optimal LC
    1. Raman retrieved LC from data
    2. Klett retrieved LC from data
    3. Raman retrieved LC from database
    4. Klett retrieved LC from database

In [31]:
## Lidar calcibration for both Klett and Raman retrival
data_cube.LidarCalibration(db_path=DATABASE_PATH)

2026-05-11 10:16:34,569 - INFO - LC retrieval: klett method
2026-05-11 10:16:34,576 - INFO - Using Retrieved Exticntion
2026-05-11 10:16:34,632 - INFO - cldFreGrp 0, Channel 532 total FR, LC_stable 113235904562719.27, LCStd 0.0022333438809804414
2026-05-11 10:16:34,639 - INFO - Using Retrieved Exticntion
2026-05-11 10:16:34,688 - INFO - cldFreGrp 0, Channel 355 total FR, LC_stable 26620168154500.37, LCStd 0.0016978405549394463
2026-05-11 10:16:34,693 - INFO - Using Retrieved Exticntion
2026-05-11 10:16:34,742 - INFO - cldFreGrp 0, Channel 532 total NR, LC_stable 7502971028367.403, LCStd 0.001417496766699034
2026-05-11 10:16:34,747 - INFO - Using Retrieved Exticntion
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\misc\helper.py:907: RuntimeWarning: Mean of empty slice
  thisMean = np.nanmean(window)
2026-05-11 10:16:34,797 - INFO - cldFreGrp 0, Channel 355 total NR, LC_stable 2647434676694.2324, LCStd 0.0017813045667396672
2026-05-11 10:16:34,804 - INFO - Using Retrieved Exticntion

In [32]:
## Display Lidar calibration constants per channel
data_cube.LCused

{'1064_total_FR': np.float64(51000118564518.48),
 '355_total_FR': np.float64(17748048015068.508),
 '355_total_NR': np.float64(2984397518140.4556),
 '532_total_FR': np.float64(117948918421429.78),
 '532_total_NR': np.float64(9799555477114.756),
 '387_total_FR': np.float64(52009299055741.99),
 '387_total_NR': np.float64(4424893374332.765),
 '607_total_FR': np.float64(308368770854909.94),
 '607_total_NR': np.float64(14951630208338.8)}

In [33]:
## Store calibration constants in database
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='raman')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='LC', method='klett')
data_cube.write_2_sql_db(db_path=str(DATABASE_PATH), parameter='DC')

2026-05-11 10:16:35,996 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-05-11 10:16:35,997 - INFO - writing LC to table: lidar_calibration_constant
2026-05-11 10:16:36,004 - INFO - 16 rows inserted into 'lidar_calibration_constant'.
2026-05-11 10:16:36,005 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-05-11 10:16:36,006 - INFO - writing LC to table: lidar_calibration_constant
2026-05-11 10:16:36,011 - INFO - 8 rows inserted into 'lidar_calibration_constant'.
2026-05-11 10:16:36,012 - INFO - writing to sqlite-db: PicassoPyDatabase.db
2026-05-11 10:16:36,013 - INFO - writing DC to table: depol_calibration_constant
2026-05-11 10:16:36,019 - INFO - 3 rows inserted into 'depol_calibration_constant'.


dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])
dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])
dict_keys(['klett', 'raman', 'klett_db', 'raman_db'])


In [34]:
## Save retrived optical profiles
write_profile2nc_file(data_cube=data_cube, prod_ls=["profiles", "NR_profiles", "OC_profiles"], collect_debug=True)

2026-05-11 10:16:36,029 - INFO - saving product: profiles
2026-05-11 10:16:36,117 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_profiles.nc
2026-05-11 10:16:36,282 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_profiles.nc
2026-05-11 10:16:36,382 - INFO - saving product: NR_profiles
2026-05-11 10:16:36,468 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_NR_profiles.nc
2026-05-11 10:16:36,595 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_NR_profiles.nc
2026-05-11 10:16:36,641 - INFO - saving product: OC_profiles
2026-05-11 10:16:36,721 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0000_0056_OC_profiles.nc
2026-05-11 10:16:36,881 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_0112_0202_OC_profiles.nc


## High Resoulution Retrievals

The following high resolution (30s) time-height data are retrieved: 

- Attenuated backscatter
- Volume depolarization
- Molecular backscatter and extinction
- Quality mask
- QuasiV1 and QuasiV2 retrievals
- Target categorization V1 and V2

In [35]:
## Highres attenuated backscatter and volume depolarization
data_cube.attBsc_volDepol()

## Highres molecular signal
data_cube.molecularHighres()

2026-05-11 10:16:36,991 - INFO - attBsc 2d retrieval
2026-05-11 10:16:37,282 - INFO - and even a 355 channel
2026-05-11 10:16:37,431 - INFO - and even a 532 channel


Exprimental, attenuated backscatter solution for 387_total_NR
G [1.] [1.]
H [0.02041] [-0.998]
polCaliEta 46.770013651088924
G [1.] [1.] H [0.02041] [-0.998] Eta 46.770013651088924 error [ 0.00016  0.01567 -0.00958] Window 1 
calculated R_t [0.95999647]


2026-05-11 10:16:37,589 - INFO - and even a 1064 channel


G [1.] [1.]
H [-0.01477] [-0.9987]
polCaliEta 12.397702207784498
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 
calculated R_t [1.02998285]
G [1.] [1.]
H [-0.02439] [-0.996]
polCaliEta 0.1511511338177992
G [1.] [1.] H [-0.02439] [-0.996] Eta 0.1511511338177992 error [ 0.00133  0.02379 -0.01205] Window 1 


2026-05-11 10:16:37,847 - INFO - voldepol 2d retrieval


calculated R_t [1.04999949]
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 
G [1.] [1.] H [0.02041] [-0.998] Eta 46.770013651088924 error [ 0.00016  0.01567 -0.00958] Window 1 
G [1.] [1.] H [-0.02439] [-0.996] Eta 0.1511511338177992 error [ 0.00133  0.02379 -0.01205] Window 1 


In [36]:
## Quality mask of signal
data_cube.estQualityMask()

['FR-total-355 nm', 'FR-cross-355 nm', 'FR-387 nm', 'FR-407 nm', 'FR-total-532 nm', 'FR-cross-532 nm', 'FR-607 nm', 'FR-total-1064 nm', 'NR-total-532 nm', 'NR-607 nm', 'NR-total-355 nm', 'NR-387 nm', 'DFOV', '1058', '1064s']
shape of quality mask (720, 4000, 15)
0 FR-total-355 nm
1 FR-cross-355 nm
2 FR-387 nm
3 FR-407 nm
4 FR-total-532 nm
5 FR-cross-532 nm
6 FR-607 nm
7 FR-total-1064 nm
8 NR-total-532 nm
9 NR-607 nm
10 NR-total-355 nm
11 NR-387 nm
12 DFOV
13 1058
14 1064s


In [37]:
## QuasiV1 retrievals and Target categorization
data_cube.quasiV1()

(720, 4000) 1 5
(720, 4000) (720, 4000)
hFullOverlap 900 120


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:112: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att) ** 2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:112: RuntimeWarning: overflow encountered in divide
  quasi_par_bsc = att_beta / (mol_att * quasi_par_att) ** 2 - molBsc
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:114: RuntimeWarning: overflow encountered in multiply
  quasi_par_ext = quasi_par_bsc * LRaer
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV1.py:111: RuntimeWarning: overflow encountered in multiply
  quasi_par_att = np.exp(-np.nancumsum(quasi_par_ext * diff_height, axis=1))


(720, 4000) 1 5
(720, 4000) (720, 4000)
hFullOverlap 800 107


c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: overflow encountered in accumulate
  return bound(*args, **kwds)


(720, 4000) 1 5
(720, 4000) (720, 4000)
hFullOverlap 800 107
G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 


c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasi.py:58: RuntimeWarning: divide by zero encountered in divide
  quasi_pdr = (vdr + 1) / (mBsc * (molDepol - vdr) / quasi_bsc / (1 + molDepol) + 1) - 1
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasi.py:73: RuntimeWarning: divide by zero encountered in divide
  ratio_par_bsc = data_cube.retrievals_highres[f'quasiBsc{version}_532_{t}_{tel}'] / \
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasi.py:73: RuntimeWarning: invalid value encountered in divide
  ratio_par_bsc = data_cube.retrievals_highres[f'quasiBsc{version}_532_{t}_{tel}'] / \
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasi.py:320: RuntimeWarning: overflow encountered in divide
  if np.min(bsc1064[hIndLargeBsc:(hIndLargeBsc + jump_hBins), iTime] / bsc1064[hIndLargeBsc, iTime]) < (1 / minAttnRatioBsc1064):


In [38]:
## QuasiV2 retrievals and Target categorization
data_cube.quasiV2()

c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:115: RuntimeWarning: divide by zero encountered in divide
  quasi_par_bsc = (att_beta_el / att_beta_ra) * quasi_par_att - molBscEl
c:\Users\buholdt\AppData\Local\miniconda3\envs\PicassoPy\Lib\site-packages\numpy\_core\fromnumeric.py:57: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:114: RuntimeWarning: overflow encountered in exp
  quasi_par_att = np.exp((1 - (wv / wv_r) ** AE) * OD_par + (OD_mol - OD_mol_r)) * molBscEl
c:\Users\buholdt\Documents\PicassoPy\tests\..\ppcpy\retrievals\quasiV2.py:112: RuntimeWarning: overflow encountered in exp
  quasi_par_att = np.exp((2 - (1064 / 607) ** AE - (1064 / 532) ** AE) * OD_par + (2 * OD_mol - OD_mol_532 - OD_mol)) * molBsc532


G [1.] [1.] H [-0.01477] [-0.9987] Eta 12.397702207784498 error [ 0.00027  0.02024 -0.00463] Window 1 


In [39]:
## Save highres retrivals
write2nc_file(data_cube=data_cube, prod_ls=["att_bsc", "NR_att_bsc", "OC_att_bsc", "vol_depol", "quasi_results", "quasi_results_V2", "target_classification", "target_classification_V2"])

2026-05-11 10:16:51,721 - INFO - saving product: att_bsc
2026-05-11 10:16:51,853 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_att_bsc.nc
2026-05-11 10:16:53,290 - INFO - saving product: NR_att_bsc
2026-05-11 10:16:53,429 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_NR_att_bsc.nc
2026-05-11 10:16:54,112 - INFO - saving product: OC_att_bsc
2026-05-11 10:16:54,201 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_OC_att_bsc.nc
2026-05-11 10:16:55,004 - INFO - saving product: vol_depol
2026-05-11 10:16:55,087 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_vol_depol.nc


removing variable: volume_depolarization_ratio_532nm_DFOV


2026-05-11 10:16:55,496 - INFO - saving product: quasi_results
2026-05-11 10:16:55,581 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_quasi_results.nc


removing variable: quality_mask_355nm
removing variable: quality_mask_532nm
removing variable: quality_mask_1064nm
removing variable: quality_mask_voldepol_532


2026-05-11 10:16:58,031 - INFO - saving product: quasi_results_V2
2026-05-11 10:16:58,135 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_quasi_results_V2.nc


removing variable: quality_mask_355nm
removing variable: quality_mask_532nm
removing variable: quality_mask_1064nm
removing variable: quality_mask_voldepol_532


2026-05-11 10:17:00,575 - INFO - saving product: target_classification
2026-05-11 10:17:00,659 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_target_classification.nc
2026-05-11 10:17:00,737 - INFO - saving product: target_classification_V2
2026-05-11 10:17:00,821 - INFO - writing to file: pollyxt_cpv\2024\08\21\20240821_pollyxt_cpv_target_classification_V2.nc
